# L3: Implicit Diffusion-Reaction

Diffusion-dominated transport couples spatial gradients to reaction in a boundary-value problem. This notebook builds the sparse gradient and divergence operators provided by pymrm, applies them to 1D steady-state and transient diffusion-reaction problems, and demonstrates Newton–Raphson iteration for nonlinear kinetics.

## Governing equations

Steady-state (BVP):

$$-D\frac{d^2 c}{dz^2} = R(c)$$

Transient:

$$\frac{\partial c}{\partial t} = D\frac{\partial^2 c}{\partial z^2} + R(c)$$

General boundary conditions (Robin form):

$$\alpha c + \beta \frac{dc}{dz} = \gamma \quad \text{at } z=0,L$$

Discretisation: gradient matrix $\mathbf{G}$ (face values from cell values), divergence matrix $\mathbf{D}$ (face fluxes → cell residuals). The diffusion term becomes $D\,\mathbf{D}\mathbf{G}\mathbf{c}$.

## PyMRM building blocks

| Function | Returns | Role |
|---|---|---|
| `construct_grad(N, dz)` | $(N{+}1)\times(N{+}2)$ sparse matrix | $dc/dz$ at faces |
| `construct_div(N, dz)` | $N\times(N{+}1)$ sparse matrix | divergence of face fluxes |
| `construct_coefficient_matrix(...)` | block-diagonal sparse | multiply $D$ per cell |
| `NumJac(f, x)` | dense/sparse Jacobian | finite-difference $\partial f/\partial x$ |
| `newton(f, x0)` | solution $x^*$ | Newton–Raphson with NumJac |
| `non_uniform_grid(N, L, f)` | node array | geometric grid stretching |

## Example 1 — Steady-state diffusion with first-order reaction

Thiele problem: slab of thickness $L$, Dirichlet BC at both faces, reaction $R = -k c$. Analytical: $c(z) = \cosh(\phi(z/L - 0.5))/\cosh(\phi/2)$ where $\phi = L\sqrt{k/D}$ is the Thiele modulus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from pymrm import construct_grad, construct_div, construct_coefficient_matrix

# Parameters
L, N = 1.0, 50
D, k = 1e-4, 1.0
c_wall = 1.0
dz = L / N
z = (np.arange(N) + 0.5) * dz

G = construct_grad(N, dz)          # (N+1) x (N+2)
Div = construct_div(N, dz)         # N x (N+1)

# Extended concentration vector: ghost cells at both ends
# Dirichlet BC: ghost = 2*c_wall - c[0] and 2*c_wall - c[-1]
def residual(c):
    c_ext = np.concatenate([[2*c_wall - c[0]], c, [2*c_wall - c[-1]]])
    flux = D * (G @ c_ext)
    return Div @ flux - k * c

# Linear system: build matrix directly
n = N
main = -2*D/dz**2 - k
off = D/dz**2
A = sp.diags([off, main, off], [-1, 0, 1], shape=(n, n), format='csr')
# BC corrections
rhs = np.zeros(n)
rhs[0] -= off * 2 * c_wall
rhs[-1] -= off * 2 * c_wall
import scipy.sparse.linalg as spla
c_num = spla.spsolve(A, rhs)

# Analytical solution
phi = L * np.sqrt(k / D)
c_exact = np.cosh(phi * (z/L - 0.5)) / np.cosh(phi/2)

plt.figure(figsize=(5.5, 3.5))
plt.plot(z, c_exact, 'k-', label=f'Exact (φ = {phi:.1f})')
plt.plot(z, c_num, 'r--', label='Numerical')
plt.xlabel('z (m)'); plt.ylabel('c / c_wall')
plt.title('Steady-state diffusion-reaction')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Max error: {np.max(np.abs(c_num - c_exact)):.2e}')

## Example 2 — Nonlinear kinetics: Newton–Raphson

Michaelis-Menten kinetics $R = -k_{\max} c / (K_m + c)$ — nonlinear BVP solved by Newton iteration.

In [ ]:
from pymrm import NumJac, newton

kmax, Km = 2.0, 0.2

def resid_nl(c):
    c_ext = np.concatenate([[2*c_wall - c[0]], c, [2*c_wall - c[-1]]])
    flux = D * (G @ c_ext)
    return Div @ flux - kmax * c / (Km + c)

c_init = np.ones(N) * c_wall
c_nl = newton(resid_nl, c_init)

# Also solve for several Thiele-equivalent phi values
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(z, c_nl, label='MM kinetics')
# Linear approximation near Km -> 0 (zero-order), Km -> inf (first-order)
for kv, label in [(kmax/Km, 'First-order approx'), (kmax, 'Zero-order approx')]:
    A2 = sp.diags([off, -2*D/dz**2 - kv, off], [-1, 0, 1], shape=(N, N), format='csr')
    rhs2 = np.zeros(N); rhs2[0] -= off * 2*c_wall; rhs2[-1] -= off * 2*c_wall
    try:
        ax.plot(z, spla.spsolve(A2, rhs2), '--', label=label)
    except Exception:
        pass
ax.set_xlabel('z (m)'); ax.set_ylabel('c (mol/m³)')
ax.set_title('Nonlinear diffusion-reaction (Michaelis-Menten)')
ax.legend(); plt.tight_layout(); plt.show()

## Summary

- Gradient $\mathbf{G}$ and divergence $\mathbf{D}$ are sparse matrices — build once, reuse
- Ghost cells encode boundary conditions: Dirichlet $c_g = 2c_w - c_1$, Neumann $c_g = c_1$, Robin: mixed
- Newton–Raphson converges quadratically; `NumJac` provides finite-difference Jacobian
- Thiele modulus $\phi = L\sqrt{k/D}$ controls internal mass-transfer limitation
- Effectiveness factor $\eta = \tanh(\phi)/\phi$ quantifies this limitation